In [4]:
import sys
import subprocess
import site
import os
import importlib

print("PDF okuma yeteneği (PyPDF2) kuruluyor...")

try:
    # PyPDF2 kütüphanesini User dizinine kur
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'PyPDF2'])
    
    user_site = site.getusersitepackages()
    if user_site not in sys.path:
        sys.path.append(user_site)
        
    importlib.invalidate_caches()
    import PyPDF2
    
    print("✅ TEST BAŞARILI: PyPDF2 başarıyla kuruldu ve entegre edildi!")
except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

PDF okuma yeteneği (PyPDF2) kuruluyor...
✅ TEST BAŞARILI: PyPDF2 başarıyla kuruldu ve entegre edildi!


In [5]:
import requests
import time
import pandas as pd
import os
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from urllib.parse import urljoin
import PyPDF2
import io

OUTPUT_FILE = "data/04_tdhd_abstracts.csv"

if not os.path.isfile(OUTPUT_FILE):
    pd.DataFrame(columns=["source", "title", "abstract", "url"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

BASE_URL = "https://www.tdhd.org/" 

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print("1. AŞAMA: Sitenin alt sayfaları taranarak PDF'ler aranıyor. Lütfen bekleyin...")

pdf_links = set()
sub_pages = [BASE_URL]

try:
    # Önce ana sayfadaki alt kategori linklerini topla
    response = requests.get(BASE_URL, headers=HEADERS, timeout=15)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    for a_tag in soup.find_all('a', href=True):
        link = a_tag['href']
        if not link.startswith('http'): 
            link = urljoin(BASE_URL, link)
        if BASE_URL in link and link not in sub_pages and not any(ext in link.lower() for ext in ['.pdf', '.jpg', '.png']):
            sub_pages.append(link)

    # Toplanan alt sayfalara girip PDF dosyalarını bul
    for page in sub_pages[:50]: # Çok derine inip kaybolmamak için ilk 50 sayfayı tarıyoruz
        try:
            p_resp = requests.get(page, headers=HEADERS, timeout=10)
            p_soup = BeautifulSoup(p_resp.content, 'html.parser')
            for a in p_soup.find_all('a', href=True):
                l = a['href']
                if '.pdf' in l.lower():
                    if not l.startswith('http'): 
                        l = urljoin(BASE_URL, l)
                    pdf_links.add(l)
        except:
            pass # Kopan linkleri geç
            
    print(f"✅ Toplam {len(pdf_links)} adet PDF belgesi (Kılavuz/Makale/Kitapçık) bulundu!\n")
    print("2. AŞAMA: PDF dosyaları indiriliyor ve içlerindeki klinik metinler kazınıyor...")
    
    dataset = []
    
    with tqdm(total=len(pdf_links), desc="PDF'ler Okunuyor") as pbar:
        for url in list(pdf_links):
            try:
                # PDF'i indirip RAM'de tut
                pdf_resp = requests.get(url, headers=HEADERS, timeout=30)
                if pdf_resp.status_code == 200:
                    pdf_file = io.BytesIO(pdf_resp.content)
                    reader = PyPDF2.PdfReader(pdf_file)
                    
                    full_text = ""
                    # PDF'in tüm sayfalarını gez ve metni birleştir
                    for page in reader.pages:
                        page_text = page.extract_text()
                        if page_text:
                            full_text += page_text + " "
                    
                    # Metni temizle (Gereksiz boşlukları ve satır atlamaları yok et)
                    clean_text = " ".join(full_text.split())
                    
                    # İçinde dişe dokunur veri varsa kaydet (Kongre kitapçıkları bazen binlerce kelime olur, bu model için harikadır)
                    if len(clean_text) > 300:
                        # PDF URL'sinden mantıklı bir başlık oluştur
                        title = url.split('/')[-1].replace('.pdf', '').replace('-', ' ').title()
                        
                        dataset.append({
                            'source': 'TDHD_PDF',
                            'title': title,
                            'abstract': clean_text,
                            'url': url
                        })
                        
                time.sleep(2) # Sunucuyu çökertmemek için bekle
                
            except Exception as e:
                pass # Şifreli veya bozuk PDF'leri atla
            
            pbar.update(1)

    if dataset:
        pd.DataFrame(dataset).to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
        print(f"\n🎉 HARİKA! {len(dataset)} adet ağır tıbbi PDF dökümanı başarıyla çözüldü ve '{OUTPUT_FILE}' dosyasına aktarıldı.")
    else:
        print("\nPDF'ler bulundu ancak içinden metin çıkarılamadı (PDF'ler 'resim' olarak taratılmış olabilir).")

except Exception as e:
    print(f"\nBağlantı hatası: {e}")

1. AŞAMA: Sitenin alt sayfaları taranarak PDF'ler aranıyor. Lütfen bekleyin...
✅ Toplam 41 adet PDF belgesi (Kılavuz/Makale/Kitapçık) bulundu!

2. AŞAMA: PDF dosyaları indiriliyor ve içlerindeki klinik metinler kazınıyor...


PDF'ler Okunuyor:   0%|          | 0/41 [00:00<?, ?it/s]


🎉 HARİKA! 34 adet ağır tıbbi PDF dökümanı başarıyla çözüldü ve 'data/04_tdhd_abstracts.csv' dosyasına aktarıldı.
